In [3]:
#chung
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

# Hàm tạo mô hình RNN
def create_RNN(hidden_units, dense_units, input_shape, activation):
    model = Sequential()
    model.add(SimpleRNN(hidden_units, input_shape=input_shape, activation=activation[0]))
    model.add(Dense(dense_units, activation=activation[1]))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Hàm chia dữ liệu X, Y theo time_steps
def get_XY(dat, time_steps):
    y_ind = np.arange(time_steps, len(dat), time_steps)
    Y = dat[y_ind]
    row_x = len(Y)
    X = dat[range(time_steps * row_x)]
    X = np.reshape(X, (row_x, time_steps, 1))
    return X, Y

# Hàm vẽ biểu đồ kết quả
def plot_result(trainY, testY, train_predict, test_predict):
    actual = np.append(trainY, testY)
    predictions = np.append(train_predict, test_predict)
    plt.figure(figsize=(15, 6))
    plt.plot(actual, label='Actual')
    plt.plot(predictions, label='Predictions')
    plt.axvline(x=len(trainY), color='r', linestyle='--')
    plt.legend()
    plt.title("Dự báo chuỗi thời gian - RNN")
    plt.show()

In [4]:
#bài 1
# Nguồn: Dự báo giá trị nhà dựa trên chuỗi lịch sử
url1 = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.csv"
df1 = pd.read_csv(url1, header=None, usecols=[13]) # Cột giá nhà cuối cùng
data1 = df1.values.astype('float32')
scaler = MinMaxScaler(feature_range=(0, 1))
data_s = scaler.fit_transform(data1).flatten()

train_d, test_d = data_s[:int(len(data_s)*0.8)], data_s[int(len(data_s)*0.8):]
tx, ty = get_XY(train_d, 12)
vx, vy = get_XY(test_d, 12)

model1 = create_RNN(3, 1, (12, 1), ['tanh', 'linear'])
model1.fit(tx, ty, epochs=20, verbose=0)
model1.save('model_cau1.h5')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
#câu 2

import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Sử dụng link dữ liệu BTC ổn định
url2 = "https://raw.githubusercontent.com/coinmetrics/data/master/csv/btc.csv"

try:
    # 1. Đọc vài dòng đầu để kiểm tra tên cột thực tế
    df_check = pd.read_csv(url2, nrows=5)
    print("Các cột có trong file:", df_check.columns.tolist())

    # 2. Tìm cột phù hợp: Ưu tiên 'price_usd_close', nếu không có thì lấy cột chứa chữ 'price'
    target_col = None
    potential_cols = [c for c in df_check.columns if 'price' in c.lower() and 'close' in c.lower()]

    if potential_cols:
        target_col = potential_cols[0]
    else:
        # Nếu vẫn không thấy, lấy cột cuối cùng thường là cột giá
        target_col = df_check.columns[-1]

    print(f"--- Đang nạp dữ liệu từ cột: {target_col} ---")

    # 3. Đọc dữ liệu chính thức
    df2 = pd.read_csv(url2, usecols=[target_col])
    df2 = df2.dropna() # Loại bỏ giá trị trống

    # 4. Chuẩn hóa dữ liệu về [0, 1]
    data2 = MinMaxScaler().fit_transform(df2.values.astype('float32')).flatten()

    # 5. Chia tập train/test (80/20)
    split2 = int(len(data2) * 0.8)
    train_data2, test_data2 = data2[:split2], data2[split2:]

    # 6. Tạo tập X, Y cho RNN (time_steps=12)
    tx2, ty2 = get_XY(train_data2, 12)

    # 7. Huấn luyện mô hình
    model2 = create_RNN(5, 1, (12, 1), ['tanh', 'tanh'])
    print("--- Đang huấn luyện mô hình RNN cho Bitcoin... ---")
    model2.fit(tx2, ty2, epochs=20, verbose=1)

    # 8. Lưu mô hình
    model2.save('model_cau2.h5')
    print("--- HOÀN THÀNH CÂU 2 ---")

except Exception as e:
    print(f"Lỗi khi xử lý: {e}")

Các cột có trong file: ['time', 'AdrActCnt', 'AdrBalCnt', 'AssetCompletionTime', 'AssetEODCompletionTime', 'BlkCnt', 'CapMVRVCur', 'CapMrktCurUSD', 'CapMrktEstUSD', 'FeeTotNtv', 'FlowInExNtv', 'FlowInExUSD', 'FlowOutExNtv', 'FlowOutExUSD', 'HashRate', 'IssTotNtv', 'IssTotUSD', 'PriceBTC', 'PriceUSD', 'ROI1yr', 'ROI30d', 'ReferenceRate', 'ReferenceRateETH', 'ReferenceRateEUR', 'ReferenceRateUSD', 'SplyCur', 'SplyExNtv', 'SplyExUSD', 'SplyExpFut10yr', 'TxCnt', 'TxTfrCnt', 'volume_reported_spot_usd_1d']
--- Đang nạp dữ liệu từ cột: volume_reported_spot_usd_1d ---
--- Đang huấn luyện mô hình RNN cho Bitcoin... ---
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.0141
Epoch 2/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0092 
Epoch 3/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0064 
Epoch 4/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0046     
Epoch 5/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0041 
Epoch 6/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0039 
Epoch 7/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0037 
Epoch 8/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0035 
Epoch 9/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0034 
Epoch 10/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0037     
Epoch 11/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0033 
Epoch 12/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0031 
Epoch 13/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0030 
Epoch 14/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0026
Epoch 15/20
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0025 
Epoch 16/

--- HOÀN THÀNH CÂU 2 ---


In [6]:
#bài 3
# Lưu ý: Household Power dùng dấu ';' và có dữ liệu rác '?'
url3 = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
df3 = pd.read_csv(url3, sep=';', usecols=[4], low_memory=False).replace('?', np.nan).dropna()
data3 = MinMaxScaler().fit_transform(df3.values[:5000].astype('float32')).flatten() # Lấy 5000 dòng mẫu

tx3, ty3 = get_XY(data3[:4000], 12)
model3 = create_RNN(4, 1, (12, 1), ['tanh', 'linear'])
model3.fit(tx3, ty3, epochs=20, verbose=0)
model3.save('model_cau3.h5')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
#bài 4

url4 = "https://raw.githubusercontent.com/mwitiderrick/stockprice/master/NSE-TATAGLOBAL.csv"
df4 = pd.read_csv(url4, usecols=[5])
data4 = MinMaxScaler().fit_transform(df4.values.astype('float32')).flatten()

tx4, ty4 = get_XY(data4[:int(len(data4)*0.8)], 12)
model4 = create_RNN(8, 1, (12, 1), ['tanh', 'linear'])
model4.fit(tx4, ty4, epochs=20, verbose=0)
model4.save('model_cau4.h5')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
from flask import Flask, request, render_template_string
from google.colab.output import eval_js

app = Flask(__name__)
# Nạp model Câu 4 để demo
curr_model = tf.keras.models.load_model('model_cau4.h5')

html = '''
<body style="font-family: sans-serif; text-align: center; padding: 50px;">
    <h2>Dự báo chuỗi thời gian RNN</h2>
    <form method="POST">
        <p>Nhập 12 giá trị gần nhất (cách nhau bằng dấu phẩy):</p>
        <input name="val" style="width: 400px" placeholder="0.1, 0.2, ...">
        <button type="submit">Dự báo giá tiếp theo</button>
    </form>
    {% if res %}<h3>Kết quả dự báo: <span style="color:red">{{ res }}</span></h3>{% endif %}
</body>
'''

@app.route('/', methods=['GET', 'POST'])
def home():
    res = None
    if request.method == 'POST':
        raw = request.form.get('val')
        # Chuyển input về định dạng (1, 12, 1) [cite: 422]
        inp = np.array([float(x) for x in raw.split(',')]).reshape(1, 12, 1)
        pred = curr_model.predict(inp)
        res = pred[0][0]
    return render_template_string(html, res=res)

if __name__ == '__main__':
    print(f"Link Web: {eval_js('google.colab.kernel.proxyPort(5000)')}")
    app.run(port=5000)

Link Web: https://5000-m-s-kkb-euw4b0-1c904zlw47gx2-b.europe-west4-0.prod.colab.dev
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:34:22] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:34:22] "GET /favicon.ico HTTP/1.1" 404 -


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step


INFO:werkzeug:127.0.0.1 - - [23/May/2026 02:34:53] "POST / HTTP/1.1" 200 -
